# Binary Classification with Spark Batch Trainer

This notebook provides a reproducible, end-to-end comparison of **XGBoost**, **CatBoost**, and **LightGBM** on a binary diabetes dataset. It follows the current public API and keeps data preparation, training, and final evaluation clearly separated.

## Learning objectives

By the end of the notebook, you will be able to:

- prepare stratified train, validation, and test datasets;
- convert only training and validation inputs to Spark DataFrames;
- configure the three supported model backends;
- inspect batch-level training history; and
- compare final results on an untouched test set.

> **Memory boundary:** Spark creates and filters batches, but each batch and the complete validation set are collected on the Python driver.

## Phase 1 — Environment and reproducibility

The bootstrap below works whether the notebook server starts from the repository root or from the `notebooks/` directory. It makes the `src/` layout explicit instead of relying on hidden environment state.

In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    """Return the nearest parent containing the project configuration."""
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not locate pyproject.toml from the current directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from pyspark.sql import SparkSession
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

from spark_batch_trainer import create_trainer

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)

RANDOM_STATE = 42
TARGET_COLUMN = "diabetes"
MAX_ROWS = 20_000  # Set to None to use the complete dataset.

## Phase 2 — Load and validate the dataset

This phase performs inexpensive quality checks before Spark or any model SDK is started. The optional row limit keeps the tutorial practical on a laptop; production experiments should size batches from measured driver capacity.

In [ ]:
dataset_path = (
    PROJECT_ROOT / "data" / "big_binary_dataset" / "binary_diabetes_dataset.csv"
)
dataset = pd.read_csv(dataset_path)

if MAX_ROWS is not None and len(dataset) > MAX_ROWS:
    dataset, _ = train_test_split(
        dataset,
        train_size=MAX_ROWS,
        stratify=dataset[TARGET_COLUMN],
        random_state=RANDOM_STATE,
    )
    dataset = dataset.reset_index(drop=True)

assert TARGET_COLUMN in dataset.columns, f"Missing target column: {TARGET_COLUMN}"
assert dataset[TARGET_COLUMN].nunique() == 2, "A binary target was expected."
assert not dataset.columns.duplicated().any(), "Duplicate column names detected."

quality_summary = pd.DataFrame(
    {
        "rows": [len(dataset)],
        "features": [dataset.shape[1] - 1],
        "missing_values": [int(dataset.isna().sum().sum())],
        "duplicate_rows": [int(dataset.duplicated().sum())],
        "positive_rate": [float(dataset[TARGET_COLUMN].mean())],
    }
)
display(quality_summary)
display(dataset.head())

### Interpretation checkpoint

Confirm that the target has two classes, review the positive-class rate, and investigate missing or duplicated observations before continuing. A stratified split preserves the observed class ratio but does not correct data-quality issues.

## Phase 3 — Create train, validation, and test splits

The validation set is used during batch training. The test set remains outside the trainer and is evaluated only after each final model has been selected.

In [ ]:
train_df, temporary_df = train_test_split(
    dataset,
    test_size=0.30,
    stratify=dataset[TARGET_COLUMN],
    random_state=RANDOM_STATE,
)
validation_df, test_df = train_test_split(
    temporary_df,
    test_size=0.50,
    stratify=temporary_df[TARGET_COLUMN],
    random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame(
    [
        {
            "split": name,
            "rows": len(frame),
            "positive_rate": frame[TARGET_COLUMN].mean(),
        }
        for name, frame in (
            ("train", train_df),
            ("validation", validation_df),
            ("test", test_df),
        )
    ]
)
display(split_summary)

## Phase 4 — Start Spark and prepare trainer inputs

Only train and validation are converted to Spark because the library consumes Spark inputs for fitting. Test data stays in pandas for bounded native-model evaluation.

In [ ]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("SparkBatchTrainerBinaryGuide")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

spark_train_df = spark.createDataFrame(train_df)
spark_validation_df = spark.createDataFrame(validation_df)

print(f"Spark version: {spark.version}")
print(f"Training rows: {spark_train_df.count():,}")
print(f"Validation rows: {spark_validation_df.count():,}")

## Phase 5 — Shared configuration and evaluation helpers

All backends use the same batch-level controls. Model hyperparameters remain backend-specific because the native SDKs use different vocabularies.

In [ ]:
TRAINING_CONFIG = {
    "num_batches": 5,
    "max_patience": 3,
    "metric_mode": "min",
    "min_delta": 1e-4,
    "use_sample_weight": False,
    "show_learning_curve": False,
    "verbose": True,
}

LEARNING_RATE_CONFIG = {
    "initial_lr": 0.05,
    "decay_rate": 0.95,
    "min_lr": 0.005,
}

In [ ]:
def prepare_native_features(frame: pd.DataFrame) -> pd.DataFrame:
    """Return model features with stable categorical dtypes."""
    features = frame.drop(columns=[TARGET_COLUMN]).copy()
    categorical_columns = features.select_dtypes(include=["object"]).columns
    features[categorical_columns] = features[categorical_columns].astype("category")
    return features


def evaluate_binary_model(name: str, model, frame: pd.DataFrame) -> dict:
    """Evaluate one native model on an in-memory holdout split."""
    features = prepare_native_features(frame)
    target = frame[TARGET_COLUMN].to_numpy()
    predictions = np.asarray(model.predict(features)).reshape(-1).astype(int)
    probabilities = np.asarray(model.predict_proba(features))[:, 1]
    return {
        "model": name,
        "accuracy": accuracy_score(target, predictions),
        "roc_auc": roc_auc_score(target, probabilities),
        "log_loss": log_loss(target, probabilities),
        "predictions": predictions,
    }


def summarize_history(trainer) -> pd.DataFrame:
    """Flatten the final metric value recorded for every batch."""
    history = trainer.get_training_history()
    return pd.DataFrame(
        {
            "batch": history.batch_numbers,
            "train_metric": [values[-1] for values in history.train_scores],
            "validation_metric": [
                values[-1] for values in history.validation_scores
            ],
        }
    )

## Phase 6 — Train XGBoost

XGBoost uses binary logistic loss. The optional learning-rate policy updates the rate between Spark batches.

In [ ]:
xgboost_trainer = create_trainer("xgboost")
xgboost_trainer.fit(
    train_dataframe=spark_train_df,
    valid_dataframe=spark_validation_df,
    target_column=TARGET_COLUMN,
    model_config={
        "objective": "binary:logistic",
        "eval_metric": "logloss",
        "n_estimators": 50,
        "learning_rate": 0.05,
        "max_depth": 6,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "random_state": RANDOM_STATE,
    },
    training_config=TRAINING_CONFIG,
    learning_rate_config=LEARNING_RATE_CONFIG,
)
xgboost_model = xgboost_trainer.get_trained_model()
display(summarize_history(xgboost_trainer))

## Phase 7 — Train CatBoost

CatBoost receives the same Spark inputs and shared training controls. `allow_writing_files=False` prevents notebook runs from creating a `catboost_info/` artifact directory.

In [ ]:
catboost_trainer = create_trainer("catboost")
catboost_trainer.fit(
    train_dataframe=spark_train_df,
    valid_dataframe=spark_validation_df,
    target_column=TARGET_COLUMN,
    model_config={
        "loss_function": "Logloss",
        "eval_metric": "Logloss",
        "iterations": 50,
        "learning_rate": 0.05,
        "depth": 6,
        "random_seed": RANDOM_STATE,
        "allow_writing_files": False,
        "verbose": False,
    },
    training_config=TRAINING_CONFIG,
)
catboost_model = catboost_trainer.get_trained_model()
display(summarize_history(catboost_trainer))

## Phase 8 — Train LightGBM

LightGBM uses binary log loss and supports the same batch-level learning-rate policy as XGBoost.

In [ ]:
lightgbm_trainer = create_trainer("lightgbm")
lightgbm_trainer.fit(
    train_dataframe=spark_train_df,
    valid_dataframe=spark_validation_df,
    target_column=TARGET_COLUMN,
    model_config={
        "objective": "binary",
        "metric": "binary_logloss",
        "n_estimators": 50,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "random_state": RANDOM_STATE,
        "verbosity": -1,
    },
    training_config=TRAINING_CONFIG,
    learning_rate_config=LEARNING_RATE_CONFIG,
)
lightgbm_model = lightgbm_trainer.get_trained_model()
display(summarize_history(lightgbm_trainer))

## Phase 9 — Final test comparison

The test set is used here for the first time. Compare ranking quality with ROC AUC, probability quality with log loss, and hard-label correctness with accuracy.

In [ ]:
evaluation_results = [
    evaluate_binary_model("XGBoost", xgboost_model, test_df),
    evaluate_binary_model("CatBoost", catboost_model, test_df),
    evaluate_binary_model("LightGBM", lightgbm_model, test_df),
]

comparison = pd.DataFrame(evaluation_results).drop(columns=["predictions"])
display(comparison.sort_values("roc_auc", ascending=False).reset_index(drop=True))

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(16, 4))
for axis, result in zip(axes, evaluation_results):
    matrix = confusion_matrix(test_df[TARGET_COLUMN], result["predictions"])
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axis)
    axis.set_title(result["model"])
    axis.set_xlabel("Predicted class")
    axis.set_ylabel("True class")
plt.suptitle("Binary test-set confusion matrices", fontweight="bold")
plt.tight_layout()
plt.show()

### Interpretation guidance

Do not select a model from one metric alone. Check whether validation and test behavior are consistent, inspect minority-class errors, and consider probability calibration when scores drive downstream decisions. The three runs are a workflow demonstration, not a complete hyperparameter search.

## Phase 10 — Resource cleanup

Stop the local Spark session explicitly when the notebook no longer needs it.

In [ ]:
spark.stop()
print("Spark session stopped.")

## Next steps

For a production experiment, remove or increase `MAX_ROWS` only after measuring driver memory, record all configurations and package versions, add calibrated probability evaluation, and persist the selected native model with its backend SDK.